# Level 0 Project Starter: Full-State Agent

Task:

> Find and sort the six cubes in the warehouse into their matching destination pads.

Level 0 allows scene_state, exact entity IDs, and entity-target go_to. The main challenge is a meaningful LLM-assisted planner, not perception.

Set both `MENLO_API_KEY` and `TOKAMAK_API_KEY` before running. The project requires a text LLM decision loop.


## How To Use This Starter

This is a minimal scaffold, not a solution. SUPPORT CODE gives you setup, wrappers, schema validation, and memory structures. STUDENT TODO sections are where your team designs the LLM decision, observation, action execution, verification, and memory update strategy.


In [ ]:
# Colab/local setup. Run this once if needed.
%pip install -q "git+https://github.com/asimovinc/hansung-menlo-robotics-workshop.git" python-dotenv opencv-python Pillow matplotlib


## Starter Scaffold

Run this cell, then edit the TODO sections as your design evolves.


In [ ]:
"""Level 0 project starter for the Menlo AI robot sorting challenge.

This file is intentionally a starter, not a solution.

Sections marked SUPPORT CODE provide small wrappers and data structures so you do
not need to rewrite workshop setup code. Sections marked STUDENT TODO are where
your project design belongs.

Level 0 rule: scene_state, exact entity IDs, and entity-target go_to are allowed.
The main challenge is still to build a meaningful LLM-assisted high-level
decision loop rather than a fixed script.
"""

import json
from dataclasses import dataclass, field
from typing import Any

from menlo_runner.scene import COLOR_TO_PAD, delivered_cube_ids, held_cube_info, visible_cubes


# ---------------------------------------------------------------------------
# SUPPORT CODE: shared task definition and required LLM decision schema
# ---------------------------------------------------------------------------
TASK = "Find and sort the six cubes in the warehouse into their matching destination pads."

DESTINATION_SIGN_RULES = {
    "red": "B",
    "green": "C",
    "blue": "D",
    "yellow": "E",
}

ALLOWED_NEXT_ACTIONS = {
    "search_cube",
    "navigate_to_cube",
    "pick_cube",
    "search_pad",
    "navigate_to_pad",
    "place_cube",
    "recover",
    "skip_target",
    "stop",
}


@dataclass
class AgentDecision:
    """Validated high-level decision returned by the LLM."""

    next_action: str
    target_color: str | None = None
    target_entity_id: str | None = None
    reason: str = ""
    recovery_strategy: str | None = None


@dataclass
class AgentMemory:
    """State your agent carries across observe-decide-act cycles."""

    delivered_count: int = 0
    delivery_limit: int | None = None
    priority_colors: list[str] = field(default_factory=list)
    held_color: str | None = None
    held_entity_id: str | None = None
    active_cube_id: str | None = None
    active_color: str | None = None
    stage: str = "need_cube"
    failed_attempts: dict[str, int] = field(default_factory=dict)
    completed_cube_ids: list[str] = field(default_factory=list)
    skipped_cube_ids: list[str] = field(default_factory=list)
    logs: list[dict[str, Any]] = field(default_factory=list)


@dataclass
class Observation:
    """Compact full-state observation passed to the LLM and action code."""

    robot_status: Any
    visible_cubes: list[dict[str, Any]]
    held_cube: dict[str, str] | None
    delivered_cube_ids: list[str]
    color_to_pad: dict[str, str]
    note: str = ""


def parse_agent_decision(text: str) -> AgentDecision | None:
    """Parse and validate the required structured LLM JSON output."""
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = stripped.strip("`")
        if stripped.lower().startswith("json"):
            stripped = stripped[4:].strip()
    if not stripped.startswith("{"):
        start = stripped.find("{")
        end = stripped.rfind("}")
        if start >= 0 and end > start:
            stripped = stripped[start : end + 1]

    try:
        data = json.loads(stripped)
    except json.JSONDecodeError:
        return None

    next_action = data.get("next_action")
    if next_action not in ALLOWED_NEXT_ACTIONS:
        return None

    target_color = data.get("target_color")
    if target_color is not None and not isinstance(target_color, str):
        return None

    target_entity_id = data.get("target_entity_id")
    if target_entity_id is not None and not isinstance(target_entity_id, str):
        return None

    return AgentDecision(
        next_action=next_action,
        target_color=target_color,
        target_entity_id=target_entity_id,
        reason=str(data.get("reason", "")),
        recovery_strategy=data.get("recovery_strategy"),
    )


def build_decision_context(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Convert full-state information into a compact text context for the LLM."""
    return {
        "task": task,
        "visible_cubes": observation.visible_cubes,
        "held_cube": observation.held_cube,
        "delivered_cube_ids": observation.delivered_cube_ids,
        "color_to_pad": observation.color_to_pad,
        "memory": {
            "delivered_count": memory.delivered_count,
            "delivery_limit": memory.delivery_limit,
            "priority_colors": memory.priority_colors,
            "held_color": memory.held_color,
            "held_entity_id": memory.held_entity_id,
            "active_cube_id": memory.active_cube_id,
            "active_color": memory.active_color,
            "stage": memory.stage,
            "failed_attempts": memory.failed_attempts,
            "completed_cube_ids": memory.completed_cube_ids,
            "skipped_cube_ids": memory.skipped_cube_ids,
        },
        "last_result": last_result,
        "note": observation.note,
    }


# ---------------------------------------------------------------------------
# SUPPORT CODE: Level 0 SDK wrappers
# ---------------------------------------------------------------------------

async def get_robot_status(ctx: Any) -> Any:
    """Read robot pose, motion status, and neck state."""
    return await ctx.state("robot_status")


async def observe_full_state(ctx: Any) -> Observation:
    """Collect a project Level 0 observation from scene_state helpers."""
    robot_status = await get_robot_status(ctx)
    cubes = [
        {
            "entity_id": cube.entity_id,
            "color": cube.color,
            "position": cube.position,
            "distance_from_robot": round(cube.distance_from_robot, 2),
        }
        for cube in await visible_cubes(ctx)
    ]
    held = await held_cube_info(ctx)
    held_dict = {"entity_id": held[0], "color": held[1]} if held else None
    delivered = await delivered_cube_ids(ctx)
    return Observation(
        robot_status=robot_status,
        visible_cubes=cubes,
        held_cube=held_dict,
        delivered_cube_ids=delivered,
        color_to_pad=dict(COLOR_TO_PAD),
    )


async def go_to_entity(ctx: Any, entity_id: str) -> Any:
    """Level 0 entity-target navigation."""
    return await ctx.invoke(
        "go_to",
        {"target": {"kind": "entity", "entity_id": entity_id}},
        timeout_s=300,
    )


async def pick_cube_by_id(ctx: Any, cube_id: str) -> Any:
    """Pick a specific cube entity after navigating close enough."""
    return await ctx.invoke(
        "pick_entity",
        {"target": {"kind": "entity", "entity_id": cube_id}},
        timeout_s=300,
    )


async def place_on_pad_by_id(ctx: Any, pad_id: str) -> Any:
    """Place the held cube on a specific pad entity."""
    return await ctx.invoke(
        "place_entity",
        {"target": {"kind": "entity", "entity_id": pad_id}},
        timeout_s=300,
    )


def result_summary(result: Any) -> dict[str, Any]:
    """Convert SDK results into a small loggable dictionary."""
    error = getattr(result, "error", None)
    status = getattr(result, "status", None)
    return {
        "status": str(status) if status is not None else None,
        "error": getattr(error, "message", None) if error else None,
    }


# ---------------------------------------------------------------------------
# STUDENT TODO: LLM decision function
# ---------------------------------------------------------------------------

async def decide_next_action(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> AgentDecision:
    """Use a text LLM to choose the next high-level action.

    TODO:
    - Build a prompt from build_decision_context(...).
    - Call menlo_runner.llm.call_llm or your approved LLM helper.
    - Require JSON with next_action, target_color, target_entity_id, and reason.
    - Validate with parse_agent_decision before execution.
    - Interpret hidden task variations, such as delivery limits or priority colors.

    The fallback below is intentionally simple. Replace it for submission.
    """
    context = build_decision_context(task, observation, memory, last_result)

    if context["held_cube"]:
        held_color = context["held_cube"]["color"]
        return AgentDecision(
            next_action="navigate_to_pad",
            target_color=held_color,
            target_entity_id=COLOR_TO_PAD.get(held_color),
            reason="Fallback: holding a cube, so navigate to its matching pad.",
        )

    remaining = [
        cube
        for cube in observation.visible_cubes
        if cube["entity_id"] not in memory.completed_cube_ids
        and cube["entity_id"] not in memory.skipped_cube_ids
    ]
    if not remaining:
        return AgentDecision(next_action="stop", reason="Fallback: no visible cubes remain.")

    cube = remaining[0]
    return AgentDecision(
        next_action="navigate_to_cube",
        target_color=cube["color"],
        target_entity_id=cube["entity_id"],
        reason="Fallback: choose the first visible cube.",
    )


# ---------------------------------------------------------------------------
# STUDENT TODO: observation, execution, verification, and memory
# ---------------------------------------------------------------------------

async def observe_world(ctx: Any, memory: AgentMemory) -> Observation:
    """Collect the current Level 0 observation for the LLM and action code.

    TODO:
    - Add natural-language task parsing results if useful.
    - Add compact scene summaries or priority notes.
    - Keep the observation small enough for repeated LLM calls.
    """
    return await observe_full_state(ctx)


async def execute_decision(
    ctx: Any,
    decision: AgentDecision,
    observation: Observation,
    memory: AgentMemory,
) -> dict[str, Any]:
    """Translate one validated LLM decision into a Level 0 robot action.

    TODO:
    - Validate target_entity_id exists and matches target_color when relevant.
    - Decide how to handle skip/recover/stop for your policy.
    - Avoid fixed scripts; let the LLM and memory choose the next high-level step.
    """
    if decision.next_action == "navigate_to_cube":
        if not decision.target_entity_id:
            return {"action": decision.next_action, "status": "failed", "reason": "missing cube id"}
        result = await go_to_entity(ctx, decision.target_entity_id)
        return {"action": "navigate_to_cube", "target": decision.target_entity_id, "result": result_summary(result)}

    if decision.next_action == "pick_cube":
        cube_id = decision.target_entity_id or memory.active_cube_id
        if not cube_id:
            return {"action": "pick_cube", "status": "failed", "reason": "missing cube id"}
        result = await pick_cube_by_id(ctx, cube_id)
        return {"action": "pick_cube", "target": cube_id, "result": result_summary(result)}

    if decision.next_action == "navigate_to_pad":
        pad_id = decision.target_entity_id
        if pad_id is None and decision.target_color:
            pad_id = COLOR_TO_PAD.get(decision.target_color)
        if not pad_id:
            return {"action": "navigate_to_pad", "status": "failed", "reason": "missing pad id"}
        result = await go_to_entity(ctx, pad_id)
        return {"action": "navigate_to_pad", "target": pad_id, "result": result_summary(result)}

    if decision.next_action == "place_cube":
        pad_id = decision.target_entity_id
        if pad_id is None and (memory.held_color or decision.target_color):
            pad_id = COLOR_TO_PAD.get(memory.held_color or decision.target_color or "")
        if not pad_id:
            return {"action": "place_cube", "status": "failed", "reason": "missing pad id"}
        result = await place_on_pad_by_id(ctx, pad_id)
        return {"action": "place_cube", "target": pad_id, "result": result_summary(result)}

    if decision.next_action in {"search_cube", "search_pad"}:
        return {"action": decision.next_action, "status": "observed_full_state"}

    if decision.next_action == "recover":
        return {"action": "recover", "status": "todo", "strategy": decision.recovery_strategy}

    return {"action": decision.next_action, "status": "no_op"}


async def verify_outcome(ctx: Any, decision: AgentDecision, action_result: dict[str, Any]) -> dict[str, Any]:
    """Check whether the last action appears to have worked.

    TODO:
    - Re-observe scene_state after important actions.
    - Check whether the robot is holding a cube after pick.
    - Check delivered_cube_ids after place.
    - Return facts the next LLM call can use for recovery.
    """
    observation = await observe_full_state(ctx)
    return {
        "decision": decision.__dict__,
        "action_result": action_result,
        "held_cube": observation.held_cube,
        "delivered_cube_ids": observation.delivered_cube_ids,
    }


def update_memory(
    memory: AgentMemory,
    observation: Observation,
    decision: AgentDecision,
    verified: dict[str, Any],
) -> None:
    """Update persistent state after each cycle.

    TODO:
    - Track active cube, held cube, delivered count, failures, and skip history.
    - Parse hidden task modifiers into delivery_limit and priority_colors.
    - Add concise logs that you can show during presentations.
    """
    held = verified.get("held_cube") or observation.held_cube
    memory.held_entity_id = held["entity_id"] if held else None
    memory.held_color = held["color"] if held else None
    memory.delivered_count = len(verified.get("delivered_cube_ids", observation.delivered_cube_ids))
    memory.completed_cube_ids = list(verified.get("delivered_cube_ids", observation.delivered_cube_ids))

    if decision.next_action == "navigate_to_cube":
        memory.active_cube_id = decision.target_entity_id
        memory.active_color = decision.target_color
        memory.stage = "near_cube"
    elif decision.next_action == "pick_cube" and memory.held_color:
        memory.stage = "holding_cube"
    elif decision.next_action == "place_cube":
        memory.stage = "need_cube"
        memory.active_cube_id = None
        memory.active_color = None

    memory.logs.append(
        {
            "observation": {
                "visible_cube_count": len(observation.visible_cubes),
                "held_cube": observation.held_cube,
                "delivered_count": memory.delivered_count,
            },
            "llm_decision": decision.__dict__,
            "verified": verified,
        }
    )


async def run_agent(ctx: Any, *, task: str = TASK, max_cycles: int = 24) -> AgentMemory:
    """Thin observe-LLM-act loop. Edit the TODO functions, not just this loop."""
    memory = AgentMemory()
    last_result: dict[str, Any] | None = None

    for cycle in range(1, max_cycles + 1):
        print(f"\n[Level 0] Cycle {cycle}")
        observation = await observe_world(ctx, memory)
        decision = await decide_next_action(task, observation, memory, last_result)
        print("LLM decision:", decision)

        if decision.next_action == "stop":
            break

        action_result = await execute_decision(ctx, decision, observation, memory)
        verified = await verify_outcome(ctx, decision, action_result)
        update_memory(memory, observation, decision, verified)
        last_result = verified

    return memory


async def run(ctx: Any) -> None:
    print(TASK)
    print("Running Level 0 full-state project starter")
    memory = await run_agent(ctx)
    print("\nRun complete.")
    print(f"Delivered count: {memory.delivered_count}")
    print("Logs:")
    for item in memory.logs:
        print(item)


## Connect Robot

Open the viewer URL in Chrome when prompted.


In [ ]:
from menlo_runner.config import load_config
from menlo_runner.context import open_robot_context

config = load_config(require_tokamak=True)
ctx = await open_robot_context(config, name_prefix="level-0-notebook")
print(ctx.viewer_url)


## Run Agent

Run after completing enough TODOs for your current experiment.


In [ ]:
memory = await run_agent(ctx)
memory.logs


In [ ]:
# Cleanup when finished.
# await ctx.close()
